# Codestral Vulkan trên Google Colab (A–Z)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 + nimpy, clone repo kèm submodule, cài Vulkan, tải tokenizer/model/adapter và chạy inference bằng Vulkan.

Repo mặc định vẫn `auto`; notebook truyền `BYBY_INFER_BACKEND=vulkan` riêng cho lần chạy này. Weights mmap từ file; merge LoRA tắt để tiết kiệm RAM/VRAM. Sửa `PROMPT` và `MAX_NEW_TOKENS` ở cell inference.


In [ ]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan SDK/runtime và shader compiler.
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','patch','pkg-config','python3-dev','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)


In [ ]:
# 2) Cài Nim 2.2.10 chính chủ, xác minh checksum, rồi cài nimpy.
import hashlib, urllib.request, tarfile

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'
if not archive.exists(): urllib.request.urlretrieve(url, archive)
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'
with tarfile.open(archive, 'r:xz') as tf: tf.extractall(TOOLS, filter='data')
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:{Path.home()}/.nimble/bin:" + os.environ['PATH']
subprocess.run(['nim','--version'], check=True)
subprocess.run(['nimble','install','-y','nimpy@0.2.1'], check=True)


In [ ]:
# 3) Clone repository tại revision đã ghim và tải submodule.
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bobbyshop-vui/nimformers-framework.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','e98e1dc2ecee2bc81e495f6fe37b79347da0f0ab'], check=True)
subprocess.run(['git','-C',str(REPO),'submodule','sync','--recursive'], check=True)
subprocess.run(['git','-C',str(REPO),'submodule','update','--init','--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO)


In [ ]:
# 4) Vulkan: kiểm tra ICD NVIDIA và xác nhận thiết bị rời (dGPU), không chấp nhận lavapipe/CPU.
# Colab thường mount Vulkan ICD của NVIDIA sẵn; apt ở cell 1 cung cấp loader + vulkaninfo.
subprocess.run(['vulkaninfo','--summary'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo','--summary'], text=True, stderr=subprocess.STDOUT)
print('
'.join(line for line in summary.splitlines() if any(k in line.lower() for k in ('device name','device type','driver name'))))
assert 'NVIDIA' in summary.upper(), 'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), 'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'


In [ ]:
# 5) Tải Codestral tokenizer/base FP4-KM và adapter. Hugging Face tự resume nếu cell bị ngắt.
subprocess.run([sys.executable,'-m','pip','-q','install','huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download, model_info

MODELS = REPO / 'test/codestral/models'
HF_DIR = MODELS / 'hf'; HF_DIR.mkdir(parents=True, exist_ok=True)
BASE_REPO = 'bobbydeveloper2014/codestral-v0.1-7b-nimq'
BASE_REV = '148dd26885b5eeef03476683f05f0a4546cd75ba'
for name in ['codestral_mamba_only.nimq','config.json','generation_config.json','params.json','special_tokens_map.json','tokenizer.json','tokenizer.model','tokenizer.model.v3','tokenizer_config.json']:
    hf_hub_download(BASE_REPO, filename=name, revision=BASE_REV, local_dir=str(HF_DIR if name.endswith('.json') or name.startswith('tokenizer.') else MODELS))
ADAPTER_REPO = 'bobbydeveloper2014/codestral-v0.1-nimq-qlora-linearattention'
ADAPTER_REV = model_info(ADAPTER_REPO).sha
adapter = hf_hub_download(ADAPTER_REPO, filename='jamba_parallel_lora.bin', revision=ADAPTER_REV, local_dir=str(MODELS))
Path(str(adapter)+'.hf_tokenizer_dir').write_text(str(HF_DIR.resolve()))
print('Base + tokenizer và adapter đã tải. Adapter revision:', ADAPTER_REV)


In [ ]:
# 6) Áp dụng patch backend, biên dịch shader Vulkan và binary inference.
subprocess.run(['make','prepare-backend'], cwd=REPO, check=True)
subprocess.run(['make','shaders'], cwd=REPO, check=True)
subprocess.run(['make','build-infer-lora'], cwd=REPO, check=True)


## Inference

Model mmap trực tiếp từ `.nimq`; LoRA không merge vào bản sao dense. `BYBY_VULKAN_DEVICE_LOCAL=auto` để backend chỉ đặt weights ở VRAM private khi Vulkan nhận diện dGPU NVIDIA. T4 dùng Tensor Core FP16 nếu kernel/driver hỗ trợ; FP4 là định dạng lưu trữ, không giả định T4 có phép tính FP4 native.


In [ ]:
# 7) Chạy inference. Chỉnh prompt/token ở đây rồi chạy lại cell.
import shlex
PROMPT = '[INST] Write a Python function for Fibonacci. [/INST]'
MAX_NEW_TOKENS = 64
model = MODELS / 'jamba_parallel_lora.bin'
base = MODELS / 'codestral_mamba_only.nimq'
assert model.exists() and base.exists() and (HF_DIR/'tokenizer.json').exists()

infer_env = os.environ.copy()
infer_env.update({
    'BYBY_INFER_BACKEND':'vulkan',
    'BYBY_MODEL_MMAP':'1',
    'BYBY_MERGE_LORA':'0',
    'BYBY_VULKAN_DEVICE_LOCAL':'auto',
    'BYBY_VULKAN_TENSOR_CORES':'1',
    'BYBY_GEMV_TG':'max',
    'BYBY_MAX_NEW_TOKENS':str(MAX_NEW_TOKENS),
    'BYBY_PREFILL_RESIDENT':'1',
})
cmd = ['./infer_lora_bin', str(model), PROMPT, '0.0', str(HF_DIR), str(base)]
print('Backend Vulkan | mmap | max tokens =', MAX_NEW_TOKENS)
print('Command:', shlex.join(cmd))
subprocess.run(cmd, cwd=REPO, env=infer_env, check=True)


### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/) · [nimpy](https://github.com/yglukhov/nimpy)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [NVIDIA Container Toolkit: graphics capability](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/docker-specialized.html)
